# 05 — Controlled evaluation, interpretability, and field deployment/QC

| Item | Definition |
|---|---|
| **Scientific purpose** | Evaluate matched synthetic experiments, expose the graph mechanism, reconstruct whole images, and deploy the trained model to the field line with conservative QC. |
| **Inputs** | Notebook-03 test realizations and normalization; Notebook-04 controlled checkpoints; Stage-01 real AVO, RGT, low-frequency elastic model, coordinates, and local wells. |
| **Outputs** | Per-realization/summary/paired metrics, documented representative figures, whole-image predictions, field consistency plots, and model/prior sensitivity products. |
| **Data availability** | Evaluation code and figure definitions are public. Checkpoints and field/private-derived arrays remain local. |
| **Local data requirements** | Completed controlled checkpoints plus authorized Stage-01 artifacts for field deployment. Numerical tables remain empty until matched artifacts exist. |
| **Software requirements** | `pip install -e ".[field,ml,notebooks]"`. |
| **Approximate runtime** | Synthetic inference: minutes per variant/realization; field tiling and sensitivity scale with checkpoints and integration steps. |
| **Pipeline position** | Final stage consuming Notebooks 01, 03, and 04. |

Field wells contributed to upstream model construction. Results are therefore described as **field deployment and QC** or **field consistency assessment**, never independent field validation. Ensemble/checkpoint/prior-cutoff spread is sensitivity, not calibrated posterior uncertainty.

In [ ]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("SAGE-AVO repository root not found; start the kernel within an installed checkout.")

ROOT = find_repository_root()

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sage_avo.config import load_config, seed_everything
from sage_avo.data import PriorDefinition, make_low_frequency_prior
from sage_avo.evaluation import (
    field_well_consistency,
    load_passing_field_calibration,
    prepare_calibrated_field_observation,
)
from sage_avo.evaluation.controlled import evaluate_controlled_ablation
from sage_avo.evaluation.inference import infer_full_realization, load_normalization
from sage_avo.evaluation.sensitivity import ensemble_sensitivity
from sage_avo.experiments.prediction import load_controlled_model, predict_controlled_variant
from sage_avo.forward import forward_avo_dense_spec, forward_specification_from_mapping
from sage_avo.forward.qc import compare_forward_outputs
from sage_avo.models import LEARNED_VARIANTS
from sage_avo.visualization import plot_inversion_comparison
from sage_avo.visualization.publication import graph_mechanism_figure

workflow_path = ROOT / "configs" / "final_training_v00332d.yaml"
paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError("Missing local configuration: configs/paths.yaml (template: configs/paths.example.yaml).")
paths = load_config(paths_file)
private_root = Path(paths["private_artifact_root"])
validation_root_text = os.getenv("SAGE_AVO_REVISION3_VALIDATION_ROOT", "").strip()
if validation_root_text:
    validation_root = Path(validation_root_text)
    workflow_path = validation_root / "configs" / "training_resolved.json"
    workflow = json.loads(workflow_path.read_text())
    dataset_dir = validation_root / "stage03" / "dataset"
    experiment_dir = validation_root / "stage04" / "sage_avo_s01_v003_stage01v003_validation8"
    figure_dir = validation_root / "figures" / "stage05"
else:
    import sys

    sys.path.insert(0, str(ROOT / "scripts"))
    from run_revision332d_final_training import _configuration as resolve_final_training

    workflow, observability = resolve_final_training()
    final_contract = load_config(workflow_path)
    dataset_dir = private_root / "stage_artifacts" / "stage03" / final_contract["immutable_dataset"] / "dataset"
    experiment_dir = private_root / "stage_artifacts" / "stage04" / "sage_avo_s01_v00332d_final_production"
    figure_dir = private_root / "figures" / "revision332d" / "stage05"
seed_everything(int(workflow["experiment"]["seed"]))
figure_dir.mkdir(parents=True, exist_ok=True)

## Part A — Controlled synthetic evaluation

The required conditions are low-frequency-prior-only, full SAGE-AVO, no-GNN, no-RGT-steering, and no-physics-loss. Learned variants are trained on the same realization split and evaluated on complete test realizations. RMSE, MAE, R², SSIM, Dice/F1, and mIoU are first computed per realization; pooled summaries and paired realization-level bootstrap intervals are secondary.

The test split is used only for final matched evaluation, never for checkpoint selection. Incomplete controlled variants remain explicitly unavailable; unmatched values are excluded from the comparison table.

In [ ]:
if not (dataset_dir / "dataset_manifest.json").exists():
    raise FileNotFoundError(f"Stage-03 dataset manifest not found: {dataset_dir / 'dataset_manifest.json'}")
checkpoints = {
    variant: experiment_dir / "runs" / variant / "best_whole_realization.pt"
    for variant in LEARNED_VARIANTS
}
checkpoints["full"] = experiment_dir / "runs" / "full" / "best_whole_realization.pt"
if validation_root_text:
    checkpoints["full"] = (
        experiment_dir
        / "runs"
        / "full_2epoch_cuda_sanity"
        / "best_whole_realization.pt"
    )
checkpoint_status = pd.DataFrame([
    {"variant": variant, "checkpoint": path.name, "available": path.exists()}
    for variant, path in checkpoints.items()
])
display(checkpoint_status)
all_checkpoints_available = bool(checkpoint_status["available"].all())

### A1. Whole-test prediction generation

In [ ]:
run_predictions = os.getenv("SAGE_AVO_RUN_EVALUATION", "0") == "1"
if run_predictions:
    if not all_checkpoints_available:
        missing = [str(path) for path in checkpoints.values() if not path.exists()]
        raise FileNotFoundError("Controlled evaluation cannot start because checkpoints are missing:\n" + "\n".join(missing))
    for variant in ("low_prior", *LEARNED_VARIANTS):
        predict_controlled_variant(
            repository=ROOT,
            config_path=workflow_path,
            config=workflow,
            dataset_directory=dataset_dir,
            experiment_directory=experiment_dir,
            variant=variant,
        )
else:
    print("Controlled evaluation is disabled (SAGE_AVO_RUN_EVALUATION=0).")
    print("Activation requires the complete matched checkpoint set from Notebook 04.")

### A2. Metrics and non-cherry-picked representative selection

In [ ]:
prediction_manifests = [
    experiment_dir / "predictions" / variant / "manifest.json"
    for variant in ("low_prior", *LEARNED_VARIANTS)
]
metrics_available = all(path.exists() for path in prediction_manifests)
if metrics_available:
    summary, per_realization, paired, representative_id = evaluate_controlled_ablation(
        experiment_directory=experiment_dir,
        dataset_directory=dataset_dir,
        bootstrap_repetitions=int(workflow["evaluation"]["bootstrap_repetitions"]),
        bootstrap_confidence=float(workflow["evaluation"]["bootstrap_confidence"]),
        seed=int(workflow["experiment"]["seed"]),
    )
    summary.to_csv(experiment_dir / "controlled_summary.csv", index=False)
    per_realization.to_csv(experiment_dir / "controlled_per_realization.csv", index=False)
    paired.to_csv(experiment_dir / "controlled_paired_improvements.csv", index=False)
    display(summary)
    display(paired)
    print("Representative test realization (median full-model Vp RMSE):", representative_id)
else:
    summary = per_realization = paired = pd.DataFrame()
    representative_id = None
    display(pd.DataFrame(columns=["variant", "domain", "metric", "mean", "std", "n_realizations"]))
    print("No controlled numerical claims are emitted until every matched prediction manifest exists.")

## Part B — Scientifically interpretable graph evidence

The representative synthetic realization is fixed by median full-model Vp RMSE—not visual appeal. A normalized forward pass through the criterion-selected checkpoint returns base edge indices, AVO-gradient edge attributes, learned final-layer mean-head `TransformerConv` attention, and node embeddings. Panel (f) displays only the strongest 15% of learned attention edges. This threshold is visualization only; all graph edges remain active in the network. The saved forward-contract JSON records all four returned tensors' dimensions/ranges.

The graph-benefit panel is

\[
|Vp_{noGNN}-Vp_{true}|-|Vp_{full}-Vp_{true}|,
\]

so positive values mean graph propagation reduces absolute error. Latent embedding norm is not used as the primary scientific evidence.

In [ ]:
if metrics_available:
    realization_path = dataset_dir / "realizations" / f"realization_{representative_id:07d}.npz"
    with np.load(realization_path) as archive:
        avo = archive["avo"]
        rgt = archive["rgt"]
        truth = archive["elastic"]
    with np.load(experiment_dir / "predictions" / "full" / f"realization_{representative_id:07d}.npz") as archive:
        full_prediction = archive["elastic"]
    with np.load(experiment_dir / "predictions" / "no_gnn" / f"realization_{representative_id:07d}.npz") as archive:
        no_gnn_prediction = archive["elastic"]

    graph_mechanism_figure(
        workflow,
        experiment_dir,
        dataset_dir,
        representative_id,
        figure_dir,
        torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    )
    print("Saved actual-attention graph mechanism figure and forward-contract JSON.")
else:
    print("Graph-benefit figure unavailable: matched full/no-GNN predictions are required.")

## Part C — Whole-image synthetic inference

In [ ]:
if metrics_available:
    with np.load(realization_path) as archive:
        low_prior = archive["low"]
    figure = plot_inversion_comparison(truth, low_prior, full_prediction)
    whole_path = figure_dir / "stage05_whole_image_synthetic.png"
    figure.savefig(whole_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Whole-image comparison unavailable: a controlled full-model prediction is required.")

## Part D — Field deployment and QC

The deployment input comprises real low/mid/high AVO stacks, Stage-01 RGT, and a field elastic background passed through the **same configured 2-Hz prior builder** used for synthetic data. The SEG-Y export axis remains described as the **configured gather-coordinate header** until acquisition/export metadata independently establish that it is true incidence angle.

Raw field AVA is never normalized directly with synthetic statistics. A versioned amplitude/phase/polarity/wavelet transfer must be recorded in a calibration manifest that satisfies explicit polarity, phase, spectrum, amplitude, percentile-overlap, and spatial-stability criteria. Missing, failed, unapproved, or forward-hash-mismatched manifests block inference. After calibration, whole-section inference uses the synthetic training normalization, common tiling/Hann stitching, and deterministic transport. Local well curves are non-blind field-consistency overlays.

In [ ]:
dataset_id = workflow["field_application"]["dataset_id"]
version = workflow["field_application"]["stage01_version"]
field_root = Path(paths["work_data_root"]) / dataset_id
field_files = {
    "avo_near": field_root / "usable" / version / "real_avo" / "AVO_low_real.npy",
    "avo_mid": field_root / "usable" / version / "real_avo" / "AVO_mid_real.npy",
    "avo_far": field_root / "usable" / version / "real_avo" / "AVO_high_real.npy",
    "low_source": field_root / "usable" / version / "elastic_background.npy",
    "rgt": field_root / "attributes" / version / "rgt_tau.npy",
    "time_ms": field_root / "usable" / version / "reg_t.npy",
    "cdp": field_root / "usable" / version / "good_cdps.npy",
    "line_xy": field_root / "usable" / version / "line_xy.npy",
}
missing_field = [path for path in field_files.values() if not path.exists()]
if missing_field:
    raise FileNotFoundError("Stage-01 field deployment channels are missing:\n" + "\n".join(map(str, missing_field)))
loaded_field = {name: np.load(path, allow_pickle=False) for name, path in field_files.items()}
field = {
    "avo": np.stack([loaded_field.pop("avo_near"), loaded_field.pop("avo_mid"), loaded_field.pop("avo_far")]),
    "low": make_low_frequency_prior(
        loaded_field.pop("low_source"),
        PriorDefinition(**workflow["field_application"]["low_frequency_prior"]),
    ),
    **loaded_field,
}
display(pd.DataFrame([{"channel": name, "shape": value.shape} for name, value in field.items()]))

fig, axes = plt.subplots(2, 4, figsize=(15, 7), constrained_layout=True)
for axis, panel, title in zip(
    axes.flat,
    [*field["avo"], field["rgt"], *field["low"], field["low"][0] - field["low"][0].mean(axis=0)],
    ["Real near AVO", "Real mid AVO", "Real far AVO", "RGT", "Low Vp", "Low Vs", "Low density", "Vp vertical variation"],
):
    axis.imshow(panel, aspect="auto", cmap="gray" if "AVO" in title else "viridis")
    axis.set_title(title); axis.set_xticks([]); axis.set_yticks([])
field_input_path = figure_dir / "stage05_field_input_contract.png"
fig.savefig(field_input_path, dpi=300, bbox_inches="tight")
plt.show()

### D1. Whole-section checkpoint inference

In [ ]:
field_prediction = field_segmentation = calibrated_field_avo = None
run_field_inference = os.getenv("SAGE_AVO_RUN_FIELD_INFERENCE", "0") == "1"
calibration_manifest = Path(
    os.getenv(
        "SAGE_AVO_FIELD_CALIBRATION_MANIFEST",
        private_root / "revision3" / "field_calibration_v003.json",
    )
)
forward_specification = forward_specification_from_mapping(workflow)
if run_field_inference and checkpoints["full"].exists():
    calibration = load_passing_field_calibration(
        calibration_manifest,
        expected_forward_specification_sha256=forward_specification.sha256,
    )
    calibrated_field_avo = prepare_calibrated_field_observation(
        field["avo"],
        calibration_manifest=calibration_manifest,
        expected_forward_specification_sha256=forward_specification.sha256,
    )
    print("Calibration record:", calibration["approved_by"], calibration["manifest_sha256"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_controlled_model("full", workflow, checkpoints["full"], device)
    field_prediction, field_segmentation = infer_full_realization(
        model,
        avo=calibrated_field_avo, low=field["low"], rgt=field["rgt"],
        normalization=load_normalization(dataset_dir),
        patch_shape=tuple(workflow["patches"]["shape"]),
        stride=tuple(workflow["patches"]["stride"]),
        steps=int(workflow["training"]["sample_steps_test"]),
        batch_size=int(workflow["training"]["batch_size"]),
        device=device,
    )
    np.savez_compressed(
        private_root / "stage_artifacts" / "stage05_field_prediction.npz",
        elastic=field_prediction, segmentation=field_segmentation,
        time_ms=field["time_ms"], cdp=field["cdp"],
    )
else:
    print("Field inference is inactive: enable it explicitly after the checkpoint and calibration contracts are satisfied.")
    print("A passing versioned calibration manifest is required; implicit identity transfer is prohibited.")

### D2. Wells, forward seismic QC, and far-angle behavior

When a prediction exists, local processed wells are overlaid in time/CDP coordinates. Exact forward-modeled near/mid/far stacks from the predicted elastic section are compared with real stacks using per-band robust amplitude fitting and correlation. Near, mid, and far statistics are reported separately; weak far-angle agreement is retained and discussed rather than hidden.

In [ ]:
if field_prediction is not None:
    well_qc, well_overlays = field_well_consistency(
        field_prediction,
        time_ms=field["time_ms"],
        line_xy=field["line_xy"],
        wells_directory=field_root / "usable" / version / "wells",
    )
    display(well_qc)
    well_qc.to_csv(private_root / "stage_artifacts" / "stage05_field_well_consistency.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    for channel, (axis, name) in enumerate(zip(axes, ("Vp", "Vs", "density"))):
        image = axis.imshow(
            field_prediction[channel], aspect="auto", cmap="viridis",
            extent=[0, field_prediction.shape[2] - 1, field["time_ms"][-1], field["time_ms"][0]],
        )
        for overlay in well_overlays:
            if overlay["channel"] == channel:
                axis.scatter(
                    np.full_like(overlay["time_ms"], overlay["trace_index"]),
                    overlay["time_ms"], c=overlay["observed"], cmap="viridis",
                    vmin=np.nanpercentile(field_prediction[channel], 2),
                    vmax=np.nanpercentile(field_prediction[channel], 98), s=2,
                )
        axis.set(title=f"Predicted {name} with processed-well overlays", xlabel="Trace", ylabel="TWT (ms)")
        fig.colorbar(image, ax=axis, shrink=0.75)
    well_path = figure_dir / "stage05_field_prediction_with_wells.png"
    fig.savefig(well_path, dpi=300, bbox_inches="tight")
    plt.show()

    modeled = forward_avo_dense_spec(*field_prediction, forward_specification).stacks
    agreement = compare_forward_outputs(calibrated_field_avo, modeled)
    forward_qc = pd.DataFrame({
        "band": ("near", "mid", "far"),
        "fitted_amplitude_scale": agreement.scale,
        "correlation": agreement.correlation,
        "normalized_rmse_after_scale": agreement.normalized_rmse,
    })
    display(forward_qc)
    forward_qc.to_csv(private_root / "stage_artifacts" / "stage05_field_forward_qc.csv", index=False)
else:
    print("Forward field QC unavailable: no field prediction has been generated.")

### D3. Model/prior sensitivity

Checkpoint ensembles, alternative justified prior cutoffs, and controlled model variants may be propagated through the same inference code. Their spread is reported as **model/prior sensitivity**, **predictive sensitivity**, or **ensemble sensitivity**. It is not a calibrated posterior uncertainty distribution.

In [ ]:
sensitivity_dir = experiment_dir / "field_sensitivity"
run_sensitivity = os.getenv("SAGE_AVO_RUN_FIELD_SENSITIVITY", "0") == "1"
if run_sensitivity:
    sensitivity_dir.mkdir(parents=True, exist_ok=True)
    if not all_checkpoints_available:
        raise FileNotFoundError("Matched controlled checkpoints are required for model-variant sensitivity.")
    if calibrated_field_avo is None:
        calibrated_field_avo = prepare_calibrated_field_observation(
            field["avo"],
            calibration_manifest=calibration_manifest,
            expected_forward_specification_sha256=forward_specification.sha256,
        )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for variant, checkpoint in checkpoints.items():
        model = load_controlled_model(variant, workflow, checkpoint, device)
        member, _ = infer_full_realization(
            model,
            avo=calibrated_field_avo, low=field["low"], rgt=field["rgt"],
            normalization=load_normalization(dataset_dir),
            patch_shape=tuple(workflow["patches"]["shape"]),
            stride=tuple(workflow["patches"]["stride"]),
            steps=int(workflow["training"]["sample_steps_test"]),
            batch_size=int(workflow["training"]["batch_size"]),
            device=device,
        )
        np.savez_compressed(sensitivity_dir / f"member_{variant}.npz", elastic=member)

sensitivity_files = sorted(sensitivity_dir.glob("member_*.npz"))
if len(sensitivity_files) >= 2:
    members = np.stack([np.load(path)["elastic"] for path in sensitivity_files])
    sensitivity = ensemble_sensitivity(members)
    print("Sensitivity members:", len(members), "shape:", sensitivity["standard_deviation"].shape)
    np.savez_compressed(sensitivity_dir / "model_variant_sensitivity_summary.npz", **sensitivity)
else:
    print("Sensitivity maps unavailable: at least two predeclared model/prior members are required.")
    print("SAGE_AVO_RUN_FIELD_SENSITIVITY=1 activates sensitivity evaluation when those members exist.")

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| controlled prediction packages | full test images per variant | Matched whole-realization benchmark predictions | metric/figure pipeline |
| per-realization/summary/paired CSVs | metric tables | Performance and paired ablation evidence | scientific reporting |
| graph mechanism figure | 2×4 high-resolution panel | AVO/RGT/edge mechanism and graph error reduction | scientific reporting |
| whole-image synthetic figure | truth/prior/prediction/error for Vp/Vs/density | Spatial inversion performance | scientific reporting |
| field prediction/QC package | whole section + coordinates + QC | Field deployment and consistency assessment | scientific reporting |
| sensitivity maps | ensemble spread | Model/prior sensitivity, not posterior uncertainty | discussion |

## Scientific checks

- Numerical tables require all matched controlled prediction manifests; absent results remain empty rather than being filled with unmatched values.
- Metrics are calculated per realization and paired by realization before aggregation.
- The representative case is chosen by median full-model Vp RMSE.
- Graph figures use returned base edges, edge attributes, learned attention, and embeddings from the actual normalized model forward pass; only the strongest 15% of attention is drawn.
- Whole-image inference uses common tiling, overlap, integration, normalization, and checkpoint rules.
- Field wells are treated as non-blind consistency overlays.
- Field inference is blocked unless a versioned calibration manifest passes and matches the forward specification hash.
- Forward QC preserves band-specific behavior, including weak far-angle agreement.
- Ensemble spread is labeled sensitivity rather than calibrated posterior uncertainty.

## Next stage

This is the final computational stage. Its artifacts support downstream scientific communication when benchmark completeness and redistribution status are recorded in the artifact index.